# ResearchMate — Evaluation: RAG vs Plain-LLM Baseline

Quantitative comparison of the full **RAG pipeline** (retrieval + grounded generation, optional Tavily web fallback)
against a **plain LLM call** (same Groq model, zero-shot, no retrieval context).

Answers are scored by an **LLM-as-a-judge** on a 1–5 rubric (correctness, groundedness, completeness, conciseness).
We also record latency, token usage, and citation behaviour.

> Run cells top-to-bottom. The corpus-embedding cell and the ~45 API calls take a few minutes.

## 0. Setup
Reuses the existing RAG logic: `backend.rag` modules + `backend.config` constants (chunk 300/30, k=5, web threshold 0.40).

In [ ]:
import json
import os
import random
import re
import sys
import time
from pathlib import Path

import numpy as np

# Make the repo root importable regardless of where the kernel is started.
REPO_ROOT = Path.cwd()
for _cand in [REPO_ROOT, REPO_ROOT.parent, REPO_ROOT.parent.parent]:
    if (_cand / "backend").is_dir():
        REPO_ROOT = _cand
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))  # noqa

from dotenv import load_dotenv
from groq import Groq

load_dotenv(REPO_ROOT / ".env")

from backend.config import (  # noqa: E402
    CHUNK_OVERLAP,
    CHUNK_SIZE,
    CORPUS_DIR,
    EMBEDDING_MODEL,
    MAX_TOKENS,
    RETRIEVAL_K,
    SYSTEM_PROMPT,
    WEB_SEARCH_THRESHOLD,
)
from backend.rag.embeddings import get_embeddings  # noqa: E402
from backend.rag.pdf_processor import chunk_text, extract_text_from_pdf  # noqa: E402
from backend.rag.web_search import search_web  # noqa: E402

MODEL = os.getenv("GROQ_MODEL", "grok-2-latest")
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

print("repo root :", REPO_ROOT)
print("corpus    :", CORPUS_DIR.name)
print("embedding :", EMBEDDING_MODEL, "| chunks:", CHUNK_SIZE, "/", CHUNK_OVERLAP, "| k:", RETRIEVAL_K)


## 1. Corpus
`load_corpus` mirrors `rag.ipynb`: it reads every `.pdf` (and `.txt`, if any) in `research_papers/`.

In [ ]:
def load_corpus(path="research_papers"):
    """Mirror of rag.ipynb load_corpus: all .pdf and .txt files."""
    path = Path(path)
    docs = []
    for fname in sorted(os.listdir(path)):
        full_path = path / fname
        if fname.lower().endswith(".pdf"):
            text = extract_text_from_pdf(str(full_path))
        elif fname.lower().endswith(".txt"):
            text = full_path.read_text(encoding="utf-8", errors="ignore")
        else:
            continue
        docs.append({"id": os.path.splitext(fname)[0], "title": fname, "text": text})
    return docs


corpus = load_corpus(CORPUS_DIR)
print(f"Loaded {len(corpus)} documents from {CORPUS_DIR.name}/")

## 2. Chunk + embed
Word chunks (300/30), `all-MiniLM-L6-v2` embeddings (384-d, L2-normalised), cosine retrieval. The embedding step can take a few minutes on CPU.

In [ ]:
chunk_records = []
for doc in corpus:
    for i, chunk in enumerate(chunk_text(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP)):
        chunk_records.append({
            "chunk_id": f"{doc['id']}_c{i}",
            "doc_id": doc["id"],
            "doc_title": doc["title"],
            "text": chunk,
        })

print(f"{len(chunk_records)} chunks from {len(corpus)} papers")

chunk_matrix = get_embeddings([c["text"] for c in chunk_records])
print("embedding matrix:", chunk_matrix.shape)

## 3. Retrieval + RAG vs baseline answersSame retrieval / context formatting / system prompt as `rag.ipynb`'s `ask()`.

In [ ]:
def retrieve(query, k=RETRIEVAL_K):
    """Cosine similarity against the chunk matrix; top-k records with scores (rag.ipynb cell 16)."""
    q_vec = get_embeddings([query])[0]
    sims = chunk_matrix @ q_vec
    top_idx = np.argsort(-sims)[:k]
    return [
        {**chunk_records[i], "score": float(sims[i])}
        for i in top_idx
        if float(sims[i]) > 0
    ]


def call_groq(**kwargs):
    """Groq completion with retry/backoff on 429 rate limits."""
    for attempt in range(6):
        try:
            return client.chat.completions.create(**kwargs)
        except Exception as exc:
            status = getattr(exc, "status_code", None)
            if status == 429 or "429" in str(exc):
                wait = 2.5 * (attempt + 1)
                print(f"  [rate limited, retrying in {wait:.1f}s]")
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("Groq kept rate-limiting after 6 attempts.")


def rag_answer(query, k=RETRIEVAL_K, with_web=True):
    """Full RAG path: retrieve -> (web fallback if low score) -> grounded LLM call."""
    t0 = time.time()
    retrieved = retrieve(query, k)
    top_score = retrieved[0]["score"] if retrieved else 0.0
    web_used = with_web and top_score < WEB_SEARCH_THRESHOLD

    context = "\n\n".join(f"[Paper: {r['doc_title']}]\n{r['text']}" for r in retrieved)
    if not context:
        context = "No relevant papers found in the corpus."

    web_results = []
    if web_used:
        web_results = search_web(query)
        if web_results:
            web_context = "\n\n".join(
                f"[Web: {r['title']}]\n{r['url']}\n{r['content'][:500]}" for r in web_results
            )
            context = (
                context
                + "\n\n--- WEB RESULTS (use only if the papers above lack the answer) ---\n\n"
                + web_context
            )

    resp = call_groq(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
        ],
    )
    return {
        "answer": resp.choices[0].message.content,
        "top1": top_score,
        "web_used": web_used,
        "web_results": web_results,
        "retrieved": [r["doc_title"] for r in retrieved],
        "latency": time.time() - t0,
        "usage": getattr(resp, "usage", None),
    }


In [ ]:
# ------- PLAIN LLM BASELINE: zero-shot, no retrieval, no context -------
BASELINE_PROMPT = (
    "You are a helpful assistant. Answer the user's question directly, clearly and concisely, "
    "based on your own knowledge. NEVER invent sources, citations, statistics, or paper names "
    "you are not sure about; if you are unsure, say so."
)


def baseline_answer(query):
    """Plain LLM call with no retrieval context (the baseline we compare against)."""
    t0 = time.time()
    resp = call_groq(
        model=MODEL,
        max_tokens=500,
        messages=[
            {"role": "system", "content": BASELINE_PROMPT},
            {"role": "user", "content": query},
        ],
    )
    return {
        "answer": resp.choices[0].message.content,
        "latency": time.time() - t0,
        "usage": getattr(resp, "usage", None),
    }


## 4. LLM-as-a-judgeOne extra Groq call per question scores **both** answers (labels A/B are randomly assigned each time to avoid position bias) on a 1–5 rubric.

In [ ]:
JUDGE_PROMPT = (
    "You are an objective, strict evaluator of AI assistant answers. "
    "You are given a QUESTION, the relevant topic(s) (what reference material EXISTS to answer it), "
    "and TWO candidate answers: ANSWER A and ANSWER B. "
    "Rate EACH answer independently on four criteria from 1 (poor) to 5 (excellent):\n"
    "- correctness: are the stated facts true, with no invented claims?\n"
    "- groundedness: are the claims supported by specific, verifiable sources cited by the answer, "
    "or by well-established knowledge? Heavily penalise fabricated citations, hallucinated statistics, "
    "fake paper/model names, and confident claims backed by nothing.\n"
    "- completeness: does the answer fully address every part of the question?\n"
    "- conciseness: is it focused and useful without irrelevant filler?\n"
    "An answer that cites concrete, verifiable sources may score higher on groundedness. "
    "Return STRICT JSON with this exact shape and nothing else:\n"
    '{"A": {"correctness": 1-5, "groundedness": 1-5, "completeness": 1-5, "conciseness": 1-5}, '
    '"B": {"correctness": 1-5, "groundedness": 1-5, "completeness": 1-5, "conciseness": 1-5}, '
    '"winner": "A" or "B" or "tie", "notes": "one or two sentences"}')


def parse_judge_json(text):
    """Robust JSON extraction (strip fences / find first JSON object / last-resort digit parse)."""
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*|```\s*$", "", text, flags=re.M).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            pass
    nums = re.findall(r"-?\d+", text)
    if len(nums) >= 9:
        keys = ["correctness", "groundedness", "completeness", "conciseness"]
        out = {"A": {}, "B": {}, "winner": "tie", "notes": text[:300]}
        for i, k in enumerate(keys):
            out["A"][k], out["B"][k] = int(nums[i]), int(nums[4 + i])
        return out
    # Unparseable — return neutral scores so the run never dies mid-way.
    keys = ["correctness", "groundedness", "completeness", "conciseness"]
    return {"A": dict.fromkeys(keys, 3), "B": dict.fromkeys(keys, 3), "winner": "tie", "notes": text[:300]}


def judge(question, topic, ans_a, ans_b):
    """Score two answers; returns {'rag_scores', 'base_scores', 'winner_system', 'notes'}."""
    swap = random.random() < 0.5  # randomise A/B to avoid position bias
    sys_a, sys_b = ("rag", "base") if not swap else ("base", "rag")
    shown_a, shown_b = (ans_a, ans_b) if not swap else (ans_b, ans_a)

    user = (
        f"QUESTION: {question}\n\n"
    f"Relevant topic(s): {topic or 'none given'}\n\n"
        f"ANSWER A:\n{shown_a}\n\n"
        f"ANSWER B:\n{shown_b}"
    )
    msgs = [{"role": "system", "content": JUDGE_PROMPT}, {"role": "user", "content": user}]
    try:
        resp = call_groq(model=MODEL, max_tokens=800, response_format={"type": "json_object"}, messages=msgs)
    except Exception:
        resp = call_groq(model=MODEL, max_tokens=800, messages=msgs)  # retry without json mode
    data = parse_judge_json(resp.choices[0].message.content)

    winner_label = data.get("winner", "tie").upper()
    if winner_label == "TIE":
        winner_system = "tie"
    else:
        winner_system = sys_a if winner_label == "A" else sys_b

    if swap:
        rag_scores, base_scores = data["B"], data["A"]
    else:
        rag_scores, base_scores = data["A"], data["B"]

    return {
        "rag_scores": rag_scores,
        "base_scores": base_scores,
        "winner_system": winner_system,
        "notes": data.get("notes", ""),
    }


## 5. Question set12 questions answerable from the corpus (one or two per paper, each with its expected source) + 3 out-of-corpus questions that trigger the Tavily web fallback.

In [ ]:
EVAL_QUERIES = [
    # in-corpus: answerable from the 9 PDFs (expected_source is a keyword in the filename)
    {"id": "q1",  "category": "in-corpus", "question": "What are the main research directions in large language models covered by the survey?", "expected_source": "Large Language"},
    {"id": "q2",  "category": "in-corpus", "question": "How does the Whale Optimization Algorithm work and how has it been improved?", "expected_source": "Whale"},
    {"id": "q3",  "category": "in-corpus", "question": "What improvements and hybridizations of the Whale Optimization Algorithm are proposed in the literature?", "expected_source": "Whale"},
    {"id": "q4",  "category": "in-corpus", "question": "What are the main multi-objective hyperparameter-optimization approaches for machine learning?", "expected_source": "multi-objective hyperparameter"},
    {"id": "q5",  "category": "in-corpus", "question": "How are machine learning and optimization methods combined in practice?", "expected_source": "Application of machine learning"},
    {"id": "q6",  "category": "in-corpus", "question": "How is Bayesian optimization used for hyperparameter tuning of machine-learning models, and where was it applied?", "expected_source": "Bayesian"},
    {"id": "q7",  "category": "in-corpus", "question": "What is the Puma optimizer and what are its applications in machine learning?", "expected_source": "Puma"},
    {"id": "q8",  "category": "in-corpus", "question": "What is retrieval-augmented generation (RAG) and for which tasks is it beneficial?", "expected_source": "Retrieval-Augmented Generation"},
    {"id": "q9",  "category": "in-corpus", "question": "Which feature-selection methods are based on optimization algorithms?", "expected_source": "Feature Selection"},
    {"id": "q10", "category": "in-corpus", "question": "How are skin lesions segmented and classified in the reviewed approach?", "expected_source": "skin lesions"},
    {"id": "q11", "category": "in-corpus", "question": "What datasets and evaluation metrics are used for skin-lesion segmentation models?", "expected_source": "skin lesions"},
    {"id": "q12", "category": "in-corpus", "question": "What open challenges and limitations does the LLM survey highlight?", "expected_source": "Large Language"},
    # out-of-corpus: forces the web-search fallback
    {"id": "q13", "category": "web-fallback", "question": "What are the most commonly cited reasons for the 2021 Suez Canal blockage?", "expected_source": ""},
    {"id": "q14", "category": "web-fallback", "question": "What are the main AI research trends expected in 2026?", "expected_source": ""},
    {"id": "q15", "category": "web-fallback", "question": "Which country has the largest population today and what is the latest estimate?", "expected_source": ""},
]
print(len(EVAL_QUERIES), "questions", "|", sum(1 for q in EVAL_QUERIES if q["category"] == "in-corpus"), "in-corpus,", sum(1 for q in EVAL_QUERIES if q["category"] == "web-fallback"), "web-fallback")

## 6. Run the evaluation~45 API calls (RAG + baseline + judge per question). Results are cached to `evaluation_results/evaluation_results.json`; set `REDO = True` to re-run.

In [ ]:
CITATION_RE = re.compile(r"\[(Paper|Web):\s")


def citation_labels(text):
    return re.findall(r"\[(Paper|Web):\s*([^\]]+)\]", text or "")


def num_citations(text):
    return len(CITATION_RE.findall(text or ""))


def expected_source_hit(answer, expected):
    if not expected:
        return None
    return any(expected.lower() in label.lower() for _, label in citation_labels(answer or ""))


REDO = False  # set True to re-run everything (re-hits the APIs)
RESULTS_DIR = REPO_ROOT / "evaluation_results"
RESULTS_DIR.mkdir(exist_ok=True)
results_path = RESULTS_DIR / "evaluation_results.json"

if not REDO and results_path.exists():
    rows = json.loads(results_path.read_text(encoding="utf-8"))
    print(f"Loaded {len(rows)} cached results from {results_path.name} (set REDO=True to re-run).")
else:
    rows = []
    for i, item in enumerate(EVAL_QUERIES, 1):
        q = item["question"]
        print(f"[{i}/{len(EVAL_QUERIES)}] ({item['category']}) {q[:60]}...")
        try:
            rag = rag_answer(q)
            base = baseline_answer(q)
            judged = judge(q, item["expected_source"], rag["answer"], base["answer"])
            rag_mean = float(np.mean(list(judged["rag_scores"].values())))
            base_mean = float(np.mean(list(judged["base_scores"].values())))
        except Exception as exc:
            print(f"    ERROR on {item['id']}: {exc}")
            rows.append({
                **item,
                "rag_score": 3.0, "base_score": 3.0, "winner": "error",
                **{f"rag_{k}": 3 for k in ["correctness", "groundedness", "completeness", "conciseness"]},
                **{f"base_{k}": 3 for k in ["correctness", "groundedness", "completeness", "conciseness"]},
                "judge_notes": f"error: {exc}", "rag_top1": None, "web_search_used": None,
                "rag_citations": 0, "base_citations": 0, "expected_hit": None,
                "rag_latency_s": None, "base_latency_s": None,
                "rag_prompt_tokens": None, "rag_completion_tokens": None,
                "base_prompt_tokens": None, "base_completion_tokens": None,
                "rag_answer": "", "base_answer": "",
            })
        else:
            rows.append({
                **item,
                "rag_score": round(rag_mean, 3),
                "base_score": round(base_mean, 3),
                "winner": judged["winner_system"],
                **{f"rag_{k}": judged["rag_scores"][k] for k in ["correctness", "groundedness", "completeness", "conciseness"]},
                **{f"base_{k}": judged["base_scores"][k] for k in ["correctness", "groundedness", "completeness", "conciseness"]},
                "judge_notes": judged["notes"],
                "rag_top1": round(rag["top1"], 3),
                "web_search_used": rag["web_used"],
                "rag_citations": num_citations(rag["answer"]),
                "base_citations": num_citations(base["answer"]),
                "expected_hit": expected_source_hit(rag["answer"], item["expected_source"]),
                "rag_latency_s": round(rag["latency"], 2),
                "base_latency_s": round(base["latency"], 2),
                "rag_prompt_tokens": getattr(rag["usage"], "prompt_tokens", None),
                "rag_completion_tokens": getattr(rag["usage"], "completion_tokens", None),
                "base_prompt_tokens": getattr(base["usage"], "prompt_tokens", None),
                "base_completion_tokens": getattr(base["usage"], "completion_tokens", None),
                "rag_answer": rag["answer"],
                "base_answer": base["answer"],
            })
        results_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
        print(f"    rag={rows[-1]['rag_score']:.2f} base={rows[-1]['base_score']:.2f} winner={rows[-1]['winner']}")
    print(f"\nSaved {len(rows)} rows to {results_path}")

import pandas as pd

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "evaluation_results.csv", index=False)
print("\nRows:", len(df))
df[["id", "category", "rag_score", "base_score", "winner", "rag_top1", "web_search_used", "rag_citations"]]

## 7. Results — summary tables

In [ ]:
dims = ["correctness", "groundedness", "completeness", "conciseness"]

overall = pd.DataFrame({
    "RAG": [df[f"rag_{d}"].mean() for d in dims] + [df["rag_score"].mean()],
    "Baseline": [df[f"base_{d}"].mean() for d in dims] + [df["base_score"].mean()],
}, index=dims + ["overall"]).round(2)
overall["RAG - Baseline"] = (overall["RAG"] - overall["Baseline"]).round(2)
print("Mean judge scores (1-5), all", len(df), "questions:\n")
print(overall.to_string())

corpus_only = df[df["category"] == "in-corpus"]
print("\n\nIn-corpus questions only (n =", len(corpus_only), "):")
in_corpus = pd.DataFrame({
    "RAG": [corpus_only[f"rag_{d}"].mean() for d in dims] + [corpus_only["rag_score"].mean()],
    "Baseline": [corpus_only[f"base_{d}"].mean() for d in dims] + [corpus_only["base_score"].mean()],
}, index=dims + ["overall"]).round(2)
in_corpus["RAG - Baseline"] = (in_corpus["RAG"] - in_corpus["Baseline"]).round(2)
print(in_corpus.to_string())

print("\n\nWinner counts (all questions):")
print(df["winner"].value_counts().reindex(["rag", "tie", "base"], fill_value=0).to_string())

print("\nCitation behaviour:")
print(f"  RAG answers with >=1 citation : {(df['rag_citations'] > 0).mean():.0%}")
print(f"  Baseline answers with >=1 cit: {(df['base_citations'] > 0).mean():.0%}")
hits = df[df["expected_hit"].notna()]
print(f"  RAG cited the EXPECTED source : {(hits['expected_hit'].astype(bool)).mean():.0%} of {len(hits)} in-corpus questions")

print("\nLatency & tokens (mean):")
lat = df[["rag_latency_s", "base_latency_s", "rag_prompt_tokens", "rag_completion_tokens", "base_prompt_tokens", "base_completion_tokens"]].mean().round(0)
print(f"  RAG     : {lat['rag_latency_s']:.1f}s  ({lat['rag_prompt_tokens']:.0f} prompt + {lat['rag_completion_tokens']:.0f} completion tokens)")
print(f"  Baseline: {lat['base_latency_s']:.1f}s  ({lat['base_prompt_tokens']:.0f} prompt + {lat['base_completion_tokens']:.0f} completion tokens)")

## 8. ChartsSaved as PNGs into `evaluation_results/`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120

w = 0.35
xs = np.arange(len(dims))
rag_m = [df[f"rag_{d}"].mean() for d in dims]
base_m = [df[f"base_{d}"].mean() for d in dims]

fig, ax = plt.subplots(figsize=(8, 4.5))
b1 = ax.bar(xs - w/2, rag_m, w, label="RAG", color="#2563eb")
b2 = ax.bar(xs + w/2, base_m, w, label="Plain LLM baseline", color="#9ca3af")
ax.bar_label(b1, fmt="%.2f", padding=2, fontsize=8)
ax.bar_label(b2, fmt="%.2f", padding=2, fontsize=8)
ax.set_xticks(xs); ax.set_xticklabels(dims)
ax.set_ylim(0, 5); ax.set_ylabel("score (1-5)")
ax.set_title("LLM-as-a-judge: mean scores per criterion")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "mean_scores_per_dimension.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(df))
ax.barh(y - w/2, df["rag_score"], w, label="RAG", color="#2563eb")
ax.barh(y + w/2, df["base_score"], w, label="Baseline", color="#9ca3af")
ax.set_yticks(y); ax.set_yticklabels([f"{r['id']} ({r['category'][:3]})" for _, r in df.iterrows()], fontsize=8)
ax.set_xlim(0, 5); ax.set_xlabel("overall score (1-5)")
ax.set_title("Overall score per question")
ax.legend(); ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "scores_per_question.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
df[["rag_latency_s", "base_latency_s"]].boxplot(ax=axes[0])
axes[0].set_xticks([1, 2], ["RAG", "Baseline"]); axes[0].set_ylabel("seconds"); axes[0].set_title("Answer latency")
df[["rag_citations", "base_citations"]].boxplot(ax=axes[1])
axes[1].set_xticks([1, 2], ["RAG", "Baseline"]); axes[1].set_ylabel("citations"); axes[1].set_title("Citations per answer")
axes[2].bar(["RAG", "Baseline"], [df["rag_score"].mean(), df["base_score"].mean()], color=["#2563eb", "#9ca3af"])
axes[2].set_ylim(0, 5); axes[2].set_title("Mean overall score"); axes[2].set_ylabel("score")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "latency_citations_overall.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("Saved artefacts:")
for p in sorted(RESULTS_DIR.iterdir()):
    print(f"  {p.name}")

## 9. Takeaways — why RAG is better (template)
Fill in the numbers from the cells above:

- **Groundedness gap** — RAG answers score ~{g} higher on groundedness because every claim is tied to a retrieved passage; the plain LLM estimates paper-specific facts from memory and hallucinates names, numbers and stats (see its answers in the CSV).
- **Sources & verifiability** — {pct}% of RAG answers carried `[Paper: …]`/`[Web: …]` citations (and cited the expected paper {hit}% of the time); the baseline produced {bpct}%.
- **Cost / latency** — grounding costs little: ~{lat}s extra and a few thousand extra prompt tokens per query for the added context.
- **On corpus-specific questions the gap is biggest**; on open web questions the Tavily fallback keeps RAG on par with the baseline.

> Caveat: the judge is the same model family as the generators (Groq). Scores are a fair automated proxy, not human annotation — spot-check a few rows in `evaluation_results.csv`.